In [ ]:
import keras
from keras import ops
from keras import layers

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf
from tensorflow.keras import backend
import random

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from scipy.fft import fft, ifft, fftfreq
import pywt
import gc
import warnings
import os

import math

In [ ]:
## Code adapted from: https://keras.io/examples/nlp/text_classification_with_transformer/
## Additional codes modified from https://github.com/samugit83/TheGradientPath/blob/master/Keras/transformers/time_series_forecast/main.py


class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=maxlen, step=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerEncoder, self).__init__()
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate)
                           for _ in range(num_layers)]
        self.dropout = layers.Dropout(rate)

    def call(self, inputs, training=False):
        x = inputs
        x = self.dropout(x, training=training)
        for layer in self.enc_layers:
            x = layer(x, training=training)
        return x

In [ ]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i + time_step)])
        y.append(data[i + time_step])
    X, y = np.array(X), np.array(y)
    # y = y.flatten()
    return X, y

In [ ]:
def model_builder(maxlen, embed_dim, num_heads, ff_dim, dim):
  inputs = layers.Input(shape=(maxlen,dim))
  x = layers.Dense(embed_dim)(inputs)
  encoder = TransformerEncoder(num_layers=8, embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim, rate=0.1)
  x = encoder(x)
  transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
  x = transformer_block(x)
  x = tf.keras.layers.Flatten()(x)
  # x = layers.GlobalAveragePooling1D()(x)
  # x = layers.Dropout(0.1)(x)
  # x = layers.Dense(64, activation="relu")(x)
  x = layers.Dropout(0.1)(x)
  x = layers.Dense(64, activation="relu")(x)
  x = layers.Dropout(0.1)(x)
  outputs = layers.Dense(dim)(x) #layers.Dense(2, activation="softmax")(x)

  model = keras.Model(inputs=inputs, outputs=outputs)
  return model

## Fine-Tuning Code:

In [ ]:
def base_transformer_model(X, y, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, seed_nr, epochs, plot_yes_no):
  random.seed(seed_nr)
  np.random.seed(seed_nr)
  tf.random.set_seed(seed_nr)
  model = model_builder(maxlen, embed_dim, num_heads, ff_dim, y.shape[1])
  X_train = X[:-test_len - valid_len]
  y_train = y[:-test_len - valid_len]

  X_valid = X[-test_len - valid_len:-test_len]
  y_valid = y[-test_len - valid_len:-test_len]

  X_test = X[-test_len:]
  y_test = y[-test_len:]

  model.compile(optimizer="adam", loss="mse", metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')])
  model.fit(X_train, y_train, batch_size=32, epochs=epochs, validation_data=(X_valid, y_valid))
  model.save_weights('model_wt.weights.h5')

  stock_data = hist[cols_list]

  prediction_short = model.predict(X)
  prediction_short_df = pd.DataFrame(prediction_short, columns = cols_list)
  prediction_short_df.index = stock_data[-len(prediction_short_df):].index
  prediction_short_df_rescaled = prediction_short_df #* (max_list - min_list) + min_list

  NRMSE_short_list = []
  for i in range(len(cols_list)):
      # rmse_value_short = np.sqrt(np.mean(((prediction_short_df_rescaled[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]) ** 2)))
      # nrmse_value_short = rmse_value_short/(stock_data[cols_list[i]].iloc[-test_len:].max()-stock_data[cols_list[i]].iloc[-test_len:].min())
      # NRMSE_short_list.append(float(nrmse_value_short))

      if plot_yes_no == 'Yes':
        # Plotting the rescaled data:
        plt.figure(figsize=(16,6))
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[:-test_len - valid_len], 'm-', label = 'Training')
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[-test_len - valid_len:-test_len], 'g-', label = 'Validation')
        plt.plot(prediction_short_df_rescaled[cols_list[i]].iloc[-test_len:], 'r-', label = '1-Day Prediction')
        plt.plot(hist[cols_list[i]], 'b-', label = 'Actual')
        plt.title(f'{stock_symbol} ({cols_list[i]})', fontsize=12)
        plt.xlabel('Date')
        plt.ylabel('Price')
        plt.legend()
        plt.show()
  # print('NRMSE (Base): ', NRMSE_short_list)
  backend.clear_session(free_memory = True)
  del model
  gc.collect()

In [ ]:
def transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, seed_nr, epochs, plot_yes_no):
  random.seed(seed_nr)
  np.random.seed(seed_nr)
  tf.random.set_seed(seed_nr)
  model = model_builder(maxlen, embed_dim, num_heads, ff_dim, y.shape[1])

  X_train = X[:-test_len - valid_len]
  y_train = y[:-test_len - valid_len]

  X_valid = X[-test_len - valid_len:-test_len]
  y_valid = y[-test_len - valid_len:-test_len]

  X_test = X[-test_len:]
  y_test = y[-test_len:]

  stock_data = hist[cols_list]


  model.load_weights('model_wt.weights.h5')
  model.compile(optimizer="adam", loss="mse", metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

  model.fit(X_train, y_train, batch_size=32, epochs=epochs, validation_data=(X_valid, y_valid))

  prediction_list = stock_data[:-test_len].values.tolist() # denoised_stock_data[:-test_len].values.tolist()
  last_value = prediction_list[-1]
  last_value[-1] = 0

  # print('Last Value:', last_value)

  prediction_short = model.predict(X_test)
  prediction_short_rescaled = prediction_short * (max_list[-test_len:] - min_list[-test_len:]) + min_list[-test_len:]
  prediction_short_total = np.array(prediction_short_rescaled).copy()
  prediction_short_total[:,:-1] = np.cumsum(prediction_short_total[:,:-1], axis=0) + last_value[:-1]

  # np.cumsum(np.array(prediction_short_rescaled),axis=0)+last_value
  prediction_short_df = pd.DataFrame(prediction_list + prediction_short_total.tolist(), columns = cols_list)
  prediction_short_df.index = stock_data[-len(prediction_short_df):].index

  prediction_long_list = []
  X_test_singular = X[-test_len:-test_len+1]
  for _ in range(test_len):

    predicted_value = model.predict(X_test_singular)

    prediction_long_list.append(predicted_value[0].tolist())


    X_test_singular = np.append(X_test_singular[0,1:],predicted_value, axis=0)
    X_test_singular = np.reshape(X_test_singular, (1, X_test_singular.shape[0],
                                              X_test_singular.shape[1]))

  prediction_long_rescaled = np.array(prediction_long_list) * (max_list[-test_len:] - min_list[-test_len:]) + min_list[-test_len:]

  prediction_long_total = np.array(prediction_long_rescaled).copy()
  prediction_long_total[:,:-1] = np.cumsum(prediction_long_total[:,:-1], axis=0) + last_value[:-1]
  # np.cumsum(np.array(prediction_long_rescaled),axis=0)+last_value
  prediction_long_df = pd.DataFrame(prediction_list + prediction_long_total.tolist(), columns = cols_list)
  prediction_long_df.index = stock_data[-len(prediction_long_df):].index

  # t_idx = np.linspace(len(y)-test_len + 1, len(y), test_len)
  # for i in range(5):
  #   plt.figure(figsize = (10, 8))
  #   plt.plot(y[:,i], 'b-')
  #   plt.plot(t_idx, prediction_short[:,i], 'r-')
  #   plt.plot(t_idx, np.array(prediction_long_list)[:,i], 'm-')
  #   plt.show()

  RMSE_short_list = []
  RMSE_long_list = []
  MAE_short_list = []
  MAE_long_list = []
  MAPE_short_list = []
  MAPE_long_list = []
  SDAPE_short_list = []
  SDAPE_long_list = []

  for i in range(len(cols_list)):
    ## RMSE calculation:
    rmse_short_value = np.sqrt(np.mean(((prediction_short_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]) ** 2)))
    rmse_long_value = np.sqrt(np.mean(((prediction_long_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]) ** 2)))
    RMSE_short_list.append(float(rmse_short_value))
    RMSE_long_list.append(float(rmse_long_value))

    ## MAE Calculation:
    mae_short_value = np.mean(np.abs(prediction_short_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]))
    mae_long_value = np.mean(np.abs(prediction_long_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:]))
    MAE_short_list.append(float(mae_short_value))
    MAE_long_list.append(float(mae_long_value))

    ## MAPE Calculation:
    mape_short_value = np.mean(np.abs((prediction_short_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:])/stock_data[cols_list[i]].iloc[-test_len:]))
    mape_long_value = np.mean(np.abs((prediction_long_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:])/stock_data[cols_list[i]].iloc[-test_len:]))
    MAPE_short_list.append(float(mape_short_value)*100)
    MAPE_long_list.append(float(mape_long_value)*100)

    ## MAPE Calculation:
    sdape_short_value = np.sqrt(np.mean((np.abs((prediction_short_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:])/stock_data[cols_list[i]].iloc[-test_len:])-mape_short_value)**2))
    sdape_long_value = np.sqrt(np.mean((np.abs((prediction_long_df[cols_list[i]].iloc[-test_len:] - stock_data[cols_list[i]].iloc[-test_len:])/stock_data[cols_list[i]].iloc[-test_len:])-mape_long_value)**2))
    SDAPE_short_list.append(float(sdape_short_value)*100)
    SDAPE_long_list.append(float(sdape_long_value)*100)

    if plot_yes_no == 'Yes':
      # ## Plotting the unscaled data:
      # plt.figure(figsize=(16,6))
      # # plt.plot(prediction_df[cols_list[i]], 'r-', label = 'Prediction')
      # plt.plot(prediction_short_df.index, y[:,i], 'b-', label = 'Actual')
      # plt.plot(prediction_short_df[cols_list[i]].iloc[:-test_len - valid_len], 'm-', label = 'Training')
      # plt.plot(prediction_short_df[cols_list[i]].iloc[-test_len - valid_len:-test_len], 'g-', label = 'Validation')
      # plt.plot(prediction_short_df[cols_list[i]].iloc[-test_len:], 'r-', label = '1-Day Prediction')
      # plt.plot(prediction_long_df[cols_list[i]].iloc[-test_len:], color='firebrick', linestyle = '-', label = f'{test_len}-Day Prediction')
      # plt.title(f'{stock_symbol} ({cols_list[i]}) - Unscaled', fontsize=12)
      # plt.xlabel('Date')
      # plt.ylabel('Price')
      # plt.legend()
      # plt.show()

      ## Plotting the rescaled data:
      plt.figure(figsize=(16,6))
      plt.plot(prediction_short_df[cols_list[i]].iloc[:-test_len - valid_len], 'm-', label = 'Training')
      plt.plot(prediction_short_df[cols_list[i]].iloc[-test_len - valid_len:-test_len], 'g-', label = 'Validation')
      plt.plot(prediction_short_df[cols_list[i]].iloc[-test_len:], 'r-', label = '1-Day Prediction')
      plt.plot(prediction_long_df[cols_list[i]].iloc[-test_len:], color='firebrick', linestyle = '-', label = f'{test_len}-Day Prediction')
      plt.plot(stock_data[cols_list[i]], 'b-', label = 'Actual')
      plt.title(f'{stock_symbol} ({cols_list[i]})', fontsize=12)
      plt.xlabel('Date')
      plt.ylabel('Price')
      plt.legend()
      plt.show()
  backend.clear_session(free_memory = True)
  del model
  gc.collect()

  return RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list, MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list

## Fourier Denoising functions:

In [ ]:
## Padding-based Fourier Transform Denoising (P-FTD), proposed by Song et al. (2021):

def padding_fourier(stock_data, quantile_percent, nr_padding_samples):
  denoised_stock_data = pd.DataFrame()

  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]].reset_index(drop=True)

    ## Creating the padding:
    X1_bar = np.sqrt(np.mean(xt[:20]))
    X2_bar = np.sqrt(np.mean(xt[-20:]))

    sigma1 = np.sqrt(np.mean(xt[:20]-X1_bar))
    sigma2 = np.sqrt(np.mean(xt[-20:]-X2_bar))

    np.random.seed(i)
    N1_list = np.random.normal(loc=0, scale=sigma1, size=nr_padding_samples)
    N2_list = np.random.normal(loc=0, scale=sigma2, size=nr_padding_samples)

    lower_pad_series = pd.Series(np.flip(N1_list.cumsum())+xt.iloc[0])
    upper_pad_series = pd.Series(N1_list.cumsum()+xt.iloc[-1])

    xt = pd.concat([lower_pad_series, xt, upper_pad_series], ignore_index=True)

    fxt = np.fft.fftn(xt)

    fxt = np.fft.fftshift(fxt)

    quantile_percent_index = int(len(fxt)*(1-quantile_percent))
    one_percent_index = int(len(fxt)*0.01)

    if stock_data.columns[i] == 'Volume':
      threshold_value = one_percent_index
    else:
      threshold_value = quantile_percent_index

    fxt_clean = fxt.copy()
    fxt_clean[:int(len(fxt_clean)/2)-threshold_value] = [0]*(int(len(fxt_clean)/2) - threshold_value)
    fxt_clean[int(len(fxt_clean)/2)+threshold_value:] = [0]*(len(fxt) - int(len(fxt_clean)/2) - threshold_value)

    # print(fxt_clean)

    plt.plot(fxt)
    plt.plot(fxt_clean)
    plt.show()
    fxt_clean = np.fft.ifftshift(fxt_clean)
    xt_clean = np.fft.ifftn(fxt_clean)

    xt_unpadded = xt_clean[nr_padding_samples:-nr_padding_samples]
    denoised_stock_data[stock_data.columns[i]] = xt_unpadded.real

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
## Our proposed Linear Drift Denoising (LDD) method:

def linear_drift_denoising(stock_data, quantile_percent):
  denoised_stock_data = pd.DataFrame()

  date_range = np.linspace(0,len(stock_data)-1,len(stock_data))
  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]]
    lin_drift_slope = (xt.iloc[-1] - xt.iloc[0])/len(xt)
    lin_drift = []
    for j in date_range:
      lin_drift.append(float(xt.iloc[0] + lin_drift_slope * j))
    xt_undrifted = xt - lin_drift
    fxt = np.fft.fft(xt_undrifted)
    if stock_data.columns[i] == 'Volume':
      threshold_value = np.quantile(np.abs(fxt), [0.99])
    else:
      threshold_value = np.quantile(np.abs(fxt), [quantile_percent])
    index = abs(fxt)> threshold_value[0]
    fxt_clean = fxt*index
    xt_clean = np.fft.ifft(fxt_clean)
    xt_clean = xt_clean.real
    xt_clean_redrifted = xt_clean + lin_drift

    denoised_stock_data[stock_data.columns[i]] = xt_clean_redrifted

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
## Our proposed Exponential Linear Drift Denoising (Exp-LDD) method:

def lindrift_exponential_denoising(stock_data, exponential_decay_rate):
  denoised_stock_data = pd.DataFrame()

  date_range = np.linspace(0,len(stock_data)-1,len(stock_data))
  for i in range(len(stock_data.columns)):
    xt = stock_data[stock_data.columns[i]]
    lin_drift_slope = (xt.iloc[-1] - xt.iloc[0])/len(xt)
    lin_drift = []
    for j in date_range:
      lin_drift.append(float(xt.iloc[0] + lin_drift_slope * j))
    xt_undrifted = xt - lin_drift
    fxt = np.fft.fft(xt_undrifted)
    fxt_unshifted = fxt.copy()
    fxt = np.fft.fftshift(fxt)
    if stock_data.columns[i]=='Volume':
      exponential_decay_rate_used = 0.8
    else:
      exponential_decay_rate_used = exponential_decay_rate
    exponential_decay_factor = exponential_decay_rate_used**np.abs(np.arange(len(fxt)) - math.floor(len(fxt)/2))
    fxt_clean = fxt*exponential_decay_factor
    fxt_clean = np.fft.ifftshift(fxt_clean)

    plt.plot(fxt_unshifted)
    plt.plot(fxt_clean)
    plt.show()

    xt_clean = np.fft.ifft(fxt_clean)
    xt_clean = xt_clean.real
    xt_clean_redrifted = xt_clean + lin_drift

    denoised_stock_data[stock_data.columns[i]] = xt_clean_redrifted

  denoised_stock_data.index = stock_data.index
  return denoised_stock_data

In [ ]:
## Our proposed Exponential Variable Denoising (Exp-VD) method:

def exponential_variable_denoising(stock_data, quantile_percent, min_max_range, smoothing_range, exponential_decay_rate):
  ## -------------- Creating the variable Min-Max range: ----------------------
  min_val = stock_data.head(min_max_range).min().tolist()
  max_val = stock_data.head(min_max_range).max().tolist()

  min_list_vals = [min_val[:] for _ in range(min_max_range)]
  max_list_vals = [max_val[:] for _ in range(min_max_range)]

  # test_len

  for i in range(len(stock_data)-min_max_range):
    min_val = stock_data.iloc[i:i+min_max_range,:].min().tolist()
    max_val = stock_data.iloc[i:i+min_max_range,:].max().tolist()
    min_list_vals.append(min_val)
    max_list_vals.append(max_val)

  # min_list = pd.DataFrame()
  # max_list = pd.DataFrame()

  min_list = pd.DataFrame(min_list_vals, columns = stock_data.columns.tolist())
  max_list = pd.DataFrame(max_list_vals, columns = stock_data.columns.tolist())

  min_list.index = stock_data.index
  max_list.index = stock_data.index

  ## Setting fixed scaling range for Volume to improve consistency:
  min_list['Volume'] = stock_data['Volume'].min()
  max_list['Volume'] = stock_data['Volume'].max()

  avg_list = (min_list+max_list)/2

  data_scaled = (stock_data - avg_list)/(max_list - min_list)

  ## -------- Smoothing the minmax scaling: --------------
  min_list_smoothed = min_list.iloc[:smoothing_range,:].values.tolist()
  max_list_smoothed = max_list.iloc[:smoothing_range,:].values.tolist()

  for i in range(len(min_list)-int(2*smoothing_range)):
    min_vals_smoothed = min_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    min_list_smoothed.append(min_vals_smoothed)

    max_vals_smoothed = max_list.iloc[i:i+int(2*smoothing_range),:].mean().values.tolist()
    max_list_smoothed.append(max_vals_smoothed)

  end_vals_min = min_list.iloc[-smoothing_range:,:].values.tolist()
  min_list_smoothed = min_list_smoothed + end_vals_min

  end_vals_max = max_list.iloc[-smoothing_range:,:].values.tolist()
  max_list_smoothed = max_list_smoothed + end_vals_max

  min_list_smoothed = pd.DataFrame(min_list_smoothed, columns = stock_data.columns.tolist())
  min_list_smoothed.index = min_list.index
  min_list_smoothed

  max_list_smoothed = pd.DataFrame(max_list_smoothed, columns = stock_data.columns.tolist())
  max_list_smoothed.index = max_list.index
  max_list_smoothed

  avg_list_smoothed = (min_list_smoothed+max_list_smoothed)/2

  ## FFT denoising of smoothed curve:
  denoised_stock_data_variable = pd.DataFrame()
  for i in range(len(data_scaled.columns)):
    ## Creating the FFT smoothed curve:
    xt = data_scaled[data_scaled.columns[i]]
    fxt = np.fft.fftn(xt)
    fxt_unshifted = fxt.copy()

    fxt = np.fft.fftshift(fxt)
    # exponential_decay_factor = exponential_decay_rate**np.abs(np.arange(len(fxt)) - len(fxt)/2)
    exponential_decay_factor = exponential_decay_rate**np.abs(np.arange(len(fxt)) - math.floor(len(fxt)/2))
    # exponential_decay_factor = exponential_decay_rate**(np.arange(len(fxt)))
    fxt_clean = fxt*exponential_decay_factor
    fxt_clean = np.fft.ifftshift(fxt_clean)

    plt.plot(fxt_unshifted)
    plt.plot(fxt_clean)
    plt.show()

    xt_clean = np.fft.ifftn(fxt_clean)
    xt_clean = xt_clean.real
    denoised_stock_data_variable[data_scaled.columns[i]] = xt_clean

  denoised_stock_data_variable.index = stock_data.index
  # for i in range(len(data_scaled.columns)):
  #   plt.figure()
  #   plt.plot(data_scaled[data_scaled.columns[i]], 'b-')
  #   plt.plot(denoised_stock_data_variable[data_scaled.columns[i]], 'r-')
  #   plt.title(data_scaled.columns[i])
  #   plt.show()

  ## ------------- Rescaling using the smoothed Min-Max scaling: ----------------

  # denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list - min_list) + avg_list
  denoised_stock_data_rescaled = denoised_stock_data_variable*(max_list_smoothed - min_list_smoothed) + avg_list_smoothed

  return denoised_stock_data_rescaled

In [ ]:
def stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no):
  test_len = 30
  memory = 20

  stock = yf.Ticker(stock_symbol)
  hist = stock.history(start=start_date, end=end_date, interval='1d')
  # hist.to_csv(f'{stock_symbol}_{start_date}_{end_date}.csv')
  stock_data = hist[cols_list]

  if denoising_method == 'None':
    denoised_stock_data = stock_data
  elif denoising_method == 'P_FTD':  ## The P-FTD method
    nr_padding_samples = 20
    denoised_stock_data = padding_fourier(stock_data.iloc[:-test_len], quantile_percent, nr_padding_samples)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'LDD':   ## The LDD method
    denoised_stock_data = linear_drift_denoising(stock_data.iloc[:-test_len], quantile_percent)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Exp_LDD':    ## The Exp-LDD method
    exponential_decay_rate = 0.95
    denoised_stock_data = lindrift_exponential_denoising(stock_data[:-test_len], exponential_decay_rate)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])
  elif denoising_method == 'Exp_VD':   ## The Exp-VD method
    min_max_range = 5
    smoothing_range = 5
    exponential_decay_rate = 0.8
    denoised_stock_data = exponential_variable_denoising(stock_data[:-test_len], quantile_percent, min_max_range, smoothing_range, exponential_decay_rate)
    denoised_stock_data = pd.concat([denoised_stock_data, stock_data.iloc[-test_len:]])


  if denoising_method != 'None' and plot_yes_no == 'Yes':
    plt.figure(figsize=(16,int(6*len(cols_list))))
    i = 0
    for col in cols_list:
      i = i+1
      plt.subplot(len(stock_data.columns), 1, i)
      plt.xlabel('Date')
      plt.ylabel(col)
      plt.plot(stock_data[col], color = 'b', linestyle = '-', label = f'Original ({col})')
      plt.plot(denoised_stock_data[col], color = 'peru', linestyle = '-', label = f'Denoised ({col})')
      plt.legend()
    plt.show()

  data_diff = denoised_stock_data[['Open', 'High', 'Low', 'Close']].diff()
  data_diff['Volume'] = denoised_stock_data['Volume']
  data_diff = data_diff.dropna()

  min_value_train = data_diff[:-test_len].min().tolist()
  max_value_train = data_diff[:-test_len].max().tolist()


  data_diff_noisy = stock_data[['Open', 'High', 'Low', 'Close']].diff()
  data_diff_noisy['Volume'] = stock_data['Volume']
  data_diff_noisy = data_diff_noisy.dropna()


  min_value_noisy = data_diff_noisy[:-test_len].min().tolist()
  max_value_noisy = data_diff_noisy[:-test_len].max().tolist()


  min_list = np.array([min_value_train[:] for _ in range(len(data_diff))])
  max_list = np.array([max_value_train[:] for _ in range(len(data_diff))])


  data_scaled = (data_diff - min_list)/(max_list - min_list)

  data_scaled = data_scaled.values.tolist()

  ## ---------------- Creating training and testing data: -----------------------
  X, y = create_dataset(data_scaled, time_step=time_step)

  print(X.shape)
  print(y.shape)


  return hist, X, y, denoised_stock_data, min_list, max_list


In [ ]:
stock_symbol = '^GSPTSE'
start_date = '2024-01-01'
end_date = '2026-01-01'
cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']
denoising_method = 'Exp_LDD'
quantile_percent = 0.95
plot_yes_no = 'Yes'

time_step = 20

hist, X, y, denoised_stock_data, min_list, max_list = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)


In [ ]:
## Stocks: ['^GSPTSE', 'SLV', 'MSFT', 'TSLA', 'CNQ.TO', 'BHP', 'BMO', 'V', 'L.TO', 'USO']

stock_symbol_list = ['^GSPTSE']

start_date_list = ['2022-01-01'] #['2022-01-01', '2022-09-01', '2023-05-01', '2024-01-01']

end_date_list = ['2024-01-01'] #['2024-01-01', '2024-09-01', '2025-05-01', '2026-01-01']

cols_list = ['Open', 'High', 'Low', 'Close', 'Volume']

## Transformer (No Denoising):

In [ ]:
denoising_method = 'None'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'


cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]

method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_Results.csv') and os.path.exists(f'Long_Transformer_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_Transformer_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_Transformer_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    # if stock_symbol_processed == Long_results[-1][0] and start_date == Long_results[-1][2]:
    #   print('STOCK NOT CHANGED')
    #   break
    hist, X, y, denoised_stock_data, min_list, max_list  = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      seed_nr = j+1
      print('Fine Tuning Run ', seed_nr)
      if j < 2:
        plot_fine_tune = 'Yes'
      else:
        plot_fine_tune = 'No'
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len,
                                                                                                              maxlen, embed_dim, num_heads, ff_dim, seed_nr, 10, plot_fine_tune)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_Transformer_{method_used}_Results.csv')
Long_results.to_csv(f'Long_Transformer_{method_used}_Results.csv')

Long_results

In [ ]:
Short_results

In [ ]:
Short_results[cols_list_mape].median()

In [ ]:
Long_results[cols_list_mape].median()

## Padding-based Fourier Transform Denoising (P-FTD):

Proposed by Song et al. (2021)

In [ ]:
denoising_method = 'P_FTD'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'


cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]

method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_Results.csv') and os.path.exists(f'Long_Transformer_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_Transformer_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_Transformer_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    # if stock_symbol_processed == Long_results[-1][0] and start_date == Long_results[-1][2]:
    #   print('STOCK NOT CHANGED')
    #   break
    hist, X, y, denoised_stock_data, min_list, max_list  = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      seed_nr = j+1
      print('Fine Tuning Run ', seed_nr)
      if j < 2:
        plot_fine_tune = 'Yes'
      else:
        plot_fine_tune = 'No'
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len,
                                                                                                              maxlen, embed_dim, num_heads, ff_dim, seed_nr, 10, plot_fine_tune)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_Transformer_{method_used}_Results.csv')
Long_results.to_csv(f'Long_Transformer_{method_used}_Results.csv')

Long_results

In [ ]:
Short_results

## Linear Drift Denoising (LDD):

In [ ]:
denoising_method = 'LDD'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'


cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]

method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_Results.csv') and os.path.exists(f'Long_Transformer_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_Transformer_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_Transformer_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    # if stock_symbol_processed == Long_results[-1][0] and start_date == Long_results[-1][2]:
    #   print('STOCK NOT CHANGED')
    #   break
    hist, X, y, denoised_stock_data, min_list, max_list  = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      seed_nr = j+1
      print('Fine Tuning Run ', seed_nr)
      if j < 2:
        plot_fine_tune = 'Yes'
      else:
        plot_fine_tune = 'No'
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len,
                                                                                                              maxlen, embed_dim, num_heads, ff_dim, seed_nr, 10, plot_fine_tune)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_Transformer_{method_used}_Results.csv')
Long_results.to_csv(f'Long_Transformer_{method_used}_Results.csv')

Long_results

In [ ]:
Short_results

## Exponential Variable Denoising (Exp-VD):

In [ ]:
denoising_method = 'Exp_VD'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'


cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]

method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_Results.csv') and os.path.exists(f'Long_Transformer_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_Transformer_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_Transformer_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    # if stock_symbol_processed == Long_results[-1][0] and start_date == Long_results[-1][2]:
    #   print('STOCK NOT CHANGED')
    #   break
    hist, X, y, denoised_stock_data, min_list, max_list  = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      seed_nr = j+1
      print('Fine Tuning Run ', seed_nr)
      if j < 2:
        plot_fine_tune = 'Yes'
      else:
        plot_fine_tune = 'No'
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len,
                                                                                                              maxlen, embed_dim, num_heads, ff_dim, seed_nr, 10, plot_fine_tune)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_Transformer_{method_used}_Results.csv')
Long_results.to_csv(f'Long_Transformer_{method_used}_Results.csv')

Long_results

In [ ]:
Short_results

## Exponential Linear Drift Denoising (Exp-LDD):

In [ ]:
denoising_method = 'Exp_LDD'
quantile_percent = 0.95

test_len = 30
memory = 20

embed_dim = 32
num_heads = 8
ff_dim = 32
maxlen = memory

valid_len = 100
test_len = 30
time_step = 20

epochs = 100

nr_runs = 50

plot_yes_no = 'Yes'
plot_base = 'No'
plot_fine_tune = 'No'


cols_list_rmse = [item+' (RMSE)' for item in cols_list]
cols_list_mae = [item+' (MAE)' for item in cols_list]
cols_list_mape = [item+' (MAPE)' for item in cols_list]
cols_list_sdape = [item+' (SDAPE)' for item in cols_list]

method_used = 'Orig' if denoising_method=='None' else denoising_method

if os.path.exists(f'Short_Transformer_{method_used}_Results.csv') and os.path.exists(f'Long_Transformer_{method_used}_Results.csv'):
  print(f'Dataset found ({denoising_method})')
  print('')
  Short_results = pd.read_csv(f'Short_Transformer_{method_used}_Results.csv')
  Long_results = pd.read_csv(f'Long_Transformer_{method_used}_Results.csv')
  Short_results = Short_results.iloc[:,1:].values.tolist()
  Long_results = Long_results.iloc[:,1:].values.tolist()
else:
  Short_results = []
  Long_results = []

for stock_symbol in stock_symbol_list:
  stock_symbol_processed = stock_symbol.replace('.', '_')
  for i in range(len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(f'{stock_symbol} ({start_date} to {end_date})')
    # if stock_symbol_processed == Long_results[-1][0] and start_date == Long_results[-1][2]:
    #   print('STOCK NOT CHANGED')
    #   break
    hist, X, y, denoised_stock_data, min_list, max_list  = stock_processor(stock_symbol, start_date, end_date, cols_list, denoising_method, quantile_percent, plot_yes_no)
    print('')
    print('*'*25, ' Transformer ', '*'*25)
    base_transformer_model(X, y, valid_len, test_len, maxlen, embed_dim, num_heads, ff_dim, 42, epochs, plot_base)
    for j in range(nr_runs):
      print('')
      print('-'*50)
      seed_nr = j+1
      print('Fine Tuning Run ', seed_nr)
      if j < 2:
        plot_fine_tune = 'Yes'
      else:
        plot_fine_tune = 'No'
      (RMSE_short_list, RMSE_long_list, MAE_short_list, MAE_long_list,
       MAPE_short_list, MAPE_long_list, SDAPE_short_list, SDAPE_long_list) = transformer_model_ls_fine_tuning(X, y, min_list, max_list, valid_len, test_len,
                                                                                                              maxlen, embed_dim, num_heads, ff_dim, seed_nr, 10, plot_fine_tune)
      print('RMSE Short Term:', RMSE_short_list)
      print('RMSE Long Term:', RMSE_long_list)
      print('MAE Short Term:', MAE_short_list)
      print('MAE Long Term:', MAE_long_list)
      print('MAPE Short Term:', MAPE_short_list)
      print('MAPE Long Term:', MAPE_long_list)
      print('SDAPE Short Term:', SDAPE_short_list)
      print('SDAPE Long Term:', SDAPE_long_list)
      Short_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_short_list + MAE_short_list + MAPE_short_list + SDAPE_short_list)
      Long_results.append([stock_symbol_processed, 'Transformer', start_date, end_date, seed_nr] + RMSE_long_list + MAE_long_list + MAPE_long_list + SDAPE_long_list)
      backend.clear_session(free_memory = True)
      gc.collect()
    os.remove('model_wt.weights.h5')


Short_results = pd.DataFrame(Short_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Long_results = pd.DataFrame(Long_results, columns = ['Stock', 'Type', 'Start Date', 'End Date', 'Seed'] + cols_list_rmse + cols_list_mae + cols_list_mape + cols_list_sdape)
Short_results.to_csv(f'Short_Transformer_{method_used}_Results.csv')
Long_results.to_csv(f'Long_Transformer_{method_used}_Results.csv')

Long_results

In [ ]:
Short_results